# Shape: profile, check and diff a lakehouse table (Python notebook)

Kernel: **Python 3.11 or 3.12** (not PySpark). Reads the Delta table with `deltalake`,
profiles it with Shape, checks it against a contract, optionally diffs it against a
baseline, writes the artifacts to `Files/<outputDir>/<table>/<timestamp>/`, shows the
HTML report, and returns a compact JSON result to the calling pipeline.

The exit value is `{table, rows, passed, violations, drifted, changes, artifactPath}`
(plus `sampled` and `truncated`). `notebookutils.notebook.exit` is the last statement
and is deliberately outside any `try`/`except`.


In [ ]:
%%configure
{"vCores": 8}


In [ ]:
# Wheel uploaded to this notebook's built-in resources folder ("Resources" > "builtin").
# Alternative once the package is on PyPI:  %pip install sqllocks-shape==0.9.0
%pip install builtin/sqllocks_shape-0.9.0-py3-none-any.whl


In [ ]:
# Parameters cell (toggle "parameter cell" in Fabric). A pipeline overrides these.
# Paths are relative to the lakehouse Files/ folder, or absolute.
tableName = "orders"      # Delta table in the default lakehouse ("schema.table" also accepted)
contractPath = ""         # optional contract JSON, for example "contracts/orders.json"
baselinePath = ""         # optional earlier .shape artifact to diff against
outputDir = "shape"       # artifacts go to Files/<outputDir>/<table>/<timestamp>/
failOnDrift = False       # True: drift against the baseline also fails the gate


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import shape

LAKEHOUSE = "/lakehouse/default"
FILES = f"{LAKEHOUSE}/Files"


def _as_bool(value) -> bool:
    """Pipeline parameters may arrive as strings."""
    if isinstance(value, str):
        return value.strip().lower() in ("1", "true", "yes", "y")
    return bool(value)


def _resolve(path: str) -> str:
    return path if os.path.isabs(path) else f"{FILES}/{path}"


failOnDrift = _as_bool(failOnDrift)
safe_name = str(tableName).replace(".", "_")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out_rel = f"{outputDir.strip('/')}/{safe_name}/{stamp}"
out_dir = Path(FILES) / out_rel
MAX_LISTED = 100  # keep the exit value small (well under 1 MB)


In [ ]:
from deltalake import DeltaTable

delta_dir = f"{LAKEHOUSE}/Tables/{str(tableName).replace('.', '/')}"
table = DeltaTable(delta_dir).to_pyarrow_table()
total_rows = table.num_rows
sampled = False  # the Python notebook profiles every row
profile = shape.profile(table, name=str(tableName))
print(f"Profiled {total_rows:,} rows x {table.num_columns} columns from {delta_dir}")


In [ ]:
violations, changes, drifted = [], [], False

if contractPath:
    check_result = shape.check(profile, _resolve(contractPath))
    violations = list(check_result.violations)

if baselinePath:
    diff_result = shape.diff(shape.load(_resolve(baselinePath)), profile)
    drifted = bool(diff_result.drifted)
    changes = list(diff_result.changes)
    if drifted and failOnDrift:
        violations.append(
            {
                "column": "*",
                "rule": "drift",
                "expected": "no drift against baseline",
                "observed": f"{len(changes)} change(s)",
            }
        )

passed = not violations


In [ ]:
out_dir.mkdir(parents=True, exist_ok=True)
shape_file = out_dir / f"{safe_name}.shape"
shape.save(profile, str(shape_file))
(out_dir / f"{safe_name}.html").write_text(profile.to_html(), encoding="utf-8")
(out_dir / f"{safe_name}.summary.json").write_text(json.dumps(profile.summary()), encoding="utf-8")

artifact_path = f"{out_rel}/{safe_name}.shape"  # relative to Files/, usable as baselinePath
print("Artifacts written to", out_dir)


In [ ]:
from IPython.display import HTML, display

display(HTML(profile.to_html()))


In [ ]:
result = {
    "table": tableName,
    "rows": total_rows,
    "passed": passed,
    "violations": violations[:MAX_LISTED],
    "drifted": drifted,
    "changes": changes[:MAX_LISTED],
    "artifactPath": artifact_path,
    "sampled": sampled,
    "truncated": len(violations) > MAX_LISTED or len(changes) > MAX_LISTED,
}
print(json.dumps(result, indent=2, default=str)[:4000])


In [ ]:
import notebookutils

notebookutils.notebook.exit(json.dumps(result, default=str))
